# Named Entity Recognition with GLiNER2

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook identifies and catalogs named entities in your text data using GLiNER2, a zero-shot named entity recognition model. Upload documents, define the entity types you want to find, and the notebook extracts every mention of people, organizations, locations, concepts, methods, or any custom category you define.

GLiNER2 requires no training and no API key. You define entity types in plain language and the model finds matching entities in your text. It runs locally on CPU with a 205M parameter model, keeping your research data private.

## Key Features

1. **Zero-Shot NER**: Define any entity type in plain language, no training required
2. **No API Key**: Runs entirely locally using the open-source GLiNER2 model
3. **Custom Entity Types**: Find people, organizations, concepts, methods, cultural practices, or anything you define
4. **Research Presets**: Built-in entity type sets for common anthropological research tasks
5. **Multi-Document Support**: Process multiple documents and track which entities appear where
6. **Frequency Analysis**: See which entities appear most often across your corpus
7. **Structured Export**: Download entity inventories as CSV or styled Excel

## Workflow

1. **Upload**: Load your text data from files (CSV, TXT, DOCX, PDF) or paste directly
2. **Configure**: Choose entity types to extract (use presets or define your own)
3. **Extract**: Run GLiNER2 across your documents
4. **Review**: Examine entity frequency tables and per-document breakdowns
5. **Export**: Download structured entity data for further analysis

## Applications

This notebook supports research that involves identifying what actors, institutions, places, and concepts appear in qualitative data. Use it to inventory entities across interview transcripts, field notes, policy documents, or media collections. The structured output can inform codebook development, network analysis, and comparative studies.

## Methodological Positioning

This notebook represents a **computational anthropology** approach. GLiNER2 identifies statistical patterns in text, not cultural meaning. Entity extraction is a starting point for analysis, not a conclusion. The researcher decides which entity types matter and how to interpret what the model finds.

**Important**: Zero-shot NER is not perfect. Review results for missed entities and false positives, especially for domain-specific concepts.

## Target Audience

Designed for anthropologists and qualitative researchers who want to systematically identify what people, places, organizations, and concepts appear across a text corpus.

## Technical Approach

The notebook uses **GLiNER2** (Fastino Labs), a unified information extraction model that performs zero-shot named entity recognition. The base model (205M parameters) runs on CPU. Text is processed in chunks to handle long documents. All processing runs locally in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2025. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

Zaratiana, Urchade, et al. 2025. "GLiNER2: Unified Schema-Based Information Extraction." arXiv preprint arXiv:2507.18546.

## Setup and Installation

Install GLiNER2 and supporting libraries. The model (~400MB) downloads on first use. In Google Colab, a GPU runtime speeds up processing but is not required (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Install required packages
!pip install gliner2 pandas openpyxl ipywidgets python-docx pypdf matplotlib -q

import re
import unicodedata
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from gliner2 import GLiNER2

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
import numpy as np

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Load model
print("Loading GLiNER2 model (first run downloads ~400MB)...")
extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
print("\u2713 Model loaded")


def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks by word count."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks if chunks else [text]


def normalize_text(text):
    """Normalize unicode characters."""
    if not isinstance(text, str): return text
    text = unicodedata.normalize('NFKC', text)
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...')]:
        text = text.replace(old, new)
    return text


def make_slug(query):
    """Create a clean filename slug."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30] or 'entities'


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)


print("\u2713 All packages loaded")
env_name = "Google Colab" if IN_COLAB else "Local Jupyter"
print(f"\u2713 Environment: {env_name}")
print("\U0001f50d Ready to configure entity extraction!")

## Data Input

Upload your text data or paste it directly. Supports CSV, TXT, DOCX, and PDF files. Reference sections are automatically stripped from academic documents.

In [ ]:
# Data Input

MIN_CHUNK_WORDS = 100

def strip_references_section(paragraphs):
    """Remove bibliography/references section from end of document."""
    ref_headings = {'references', 'bibliography', 'works cited', 'literature cited',
                    'sources cited', 'endnotes', 'notes and references'}
    for i, para in enumerate(paragraphs):
        if para.strip().lower().rstrip(':.') in ref_headings and len(para.split()) <= 4:
            return paragraphs[:i]
    return paragraphs

def clean_citation_noise(text):
    """Remove inline citation patterns."""
    text = re.sub(r'\([A-Z][a-z]+(?:\s+(?:and|&)\s+[A-Z][a-z]+)*(?:\s+et\s+al\.?)?,?\s*(?:19|20)\d{2}[a-z]?\)', '', text)
    text = re.sub(r'\b(?:19|20)\d{2}[a-z]?\b', '', text)
    text = re.sub(r'https?://doi\.org/[^\s]+', '', text)
    return text

def is_junk_segment(text):
    """Detect non-prose content."""
    words = text.split()
    if len(words) < 10: return True
    return sum(1 for w in words if re.match(r'^[a-zA-Z]{2,}', w)) / len(words) < 0.5

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

input_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
input_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f50d Named Entity Recognition</h2>'
input_html += '<p><strong>Welcome!</strong> This notebook identifies people, places, organizations, concepts, and other entities in your text using GLiNER2.</p>'
input_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
input_html += '<ol>'
input_html += '<li><strong>Load:</strong> Upload files (CSV, TXT, DOCX, PDF) or paste text below</li>'
input_html += '<li><strong>Configure:</strong> Choose entity types to extract</li>'
input_html += '<li><strong>Extract:</strong> Run GLiNER2 to find entities</li>'
input_html += '<li><strong>Export:</strong> Download structured results</li>'
input_html += '</ol>'
input_html += '</div>'

input_instructions = widgets.HTML(value=input_html)

input_method = widgets.ToggleButtons(
    options=[('Upload Files', 'upload'), ('Paste Text', 'paste')],
    value='upload',
    style={'button_width': '150px'}
)

csv_col = widgets.Text(value='text', description='Text column:', placeholder='Column name containing text (for CSV)', layout=layout, style=style)
csv_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">For CSV files, specify which column contains the text to analyze.</p>')

paste_area = widgets.Textarea(value='', placeholder='Paste text here. Separate documents with blank lines.', layout=widgets.Layout(width='100%', height='200px'))

documents = []
doc_names = []

upload_box = widgets.VBox([csv_col, csv_help])
paste_box = widgets.VBox([paste_area])
paste_box.layout.display = 'none'

def on_method_change(change):
    if change['new'] == 'upload':
        upload_box.layout.display = ''
        paste_box.layout.display = 'none'
    else:
        upload_box.layout.display = 'none'
        paste_box.layout.display = ''
input_method.observe(on_method_change, names='value')

load_btn = widgets.Button(description='Load Documents', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='20px 0'))
load_out = widgets.Output()

def on_load(_):
    global documents, doc_names
    load_out.clear_output()
    documents, doc_names = [], []

    with load_out:
        if input_method.value == 'paste':
            raw = paste_area.value.strip()
            if not raw:
                print("\u26a0\ufe0f Please paste some text.")
                return
            docs = [d.strip() for d in re.split(r'\n\s*\n', raw) if d.strip()]
            if len(docs) == 1:
                docs = [d.strip() for d in raw.split('\n') if d.strip()]
            documents = [d for d in docs if not is_junk_segment(d)]
            doc_names = [f'doc_{i+1}' for i in range(len(documents))]
            print(f"\u2713 Loaded {len(documents)} text segments from pasted text")

        elif input_method.value == 'upload':
            if IN_COLAB:
                from google.colab import files as colab_files
                uploaded = colab_files.upload()
                if not uploaded:
                    print("\u26a0\ufe0f No files uploaded.")
                    return
                for fname, content in uploaded.items():
                    if fname.endswith('.csv'):
                        import io
                        df = pd.read_csv(io.BytesIO(content))
                        col = csv_col.value.strip()
                        if col not in df.columns:
                            print(f"\u26a0\ufe0f Column '{col}' not found. Available: {list(df.columns)}")
                            return
                        docs = df[col].dropna().astype(str).tolist()
                        documents.extend(docs)
                        doc_names.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.txt'):
                        text = content.decode('utf-8', errors='ignore')
                        paras = [d.strip() for d in text.split('\n') if d.strip()]
                        paras = strip_references_section(paras)
                        paras = [clean_citation_noise(p) for p in paras]
                        docs = [p for p in paras if not is_junk_segment(p)]
                        documents.extend(docs)
                        doc_names.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.docx'):
                        import docx as _docx, io
                        doc = _docx.Document(io.BytesIO(content))
                        paras = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
                        paras = strip_references_section(paras)
                        paras = [clean_citation_noise(p) for p in paras]
                        docs = [p for p in paras if not is_junk_segment(p)]
                        documents.extend(docs)
                        doc_names.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.pdf'):
                        import io
                        try:
                            from pypdf import PdfReader
                        except ImportError:
                            from PyPDF2 import PdfReader
                        reader = PdfReader(io.BytesIO(content))
                        text = '\n'.join(page.extract_text() or '' for page in reader.pages)
                        paras = [d.strip() for d in text.split('\n') if d.strip()]
                        paras = strip_references_section(paras)
                        paras = [clean_citation_noise(p) for p in paras]
                        docs = [p for p in paras if not is_junk_segment(p)]
                        documents.extend(docs)
                        doc_names.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    else:
                        print(f"\u26a0\ufe0f Unsupported format: {fname}")
                        continue
                print(f"\u2713 Loaded {len(documents)} text segments from {len(uploaded)} file(s)")
            else:
                print("\u26a0\ufe0f File upload works in Colab. Locally, use Paste Text.")

        if documents:
            avg = np.mean([len(d.split()) for d in documents])
            print(f"   Average segment length: {avg:.0f} words")

load_btn.on_click(on_load)

display(widgets.VBox([
    input_instructions,
    input_method,
    upload_box,
    paste_box,
    load_btn,
    load_out,
]))

## Entity Type Configuration

Define what entity types to extract. Use a preset or define your own. GLiNER2 works best when entity types include a brief description.

In [ ]:
# Entity Type Configuration

PRESETS = {
    'General Research': {
        'person': 'Specific named individuals (first and/or last name)',
        'organization': 'Named companies, institutions, NGOs, government agencies',
        'location': 'Named places, cities, regions, countries',
        'concept': 'Named theories, frameworks, paradigms, and key technical terms',
    },
    'Ethnographic Fieldwork': {
        'person': 'Specific named individuals (first and/or last name)',
        'organization': 'Named institutions, agencies, community organizations',
        'location': 'Named places, neighborhoods, regions, field sites',
        'cultural practice': 'Named rituals, customs, traditions, social practices',
        'kinship term': 'Named family roles, clan names, social categories',
        'material object': 'Named tools, artifacts, foods, technologies',
    },
    'Policy & Applied': {
        'person': 'Specific named individuals (first and/or last name)',
        'organization': 'Named government agencies, NGOs, corporations',
        'location': 'Named places, jurisdictions, regions',
        'policy': 'Named laws, regulations, programs, initiatives',
        'stakeholder group': 'Named communities, populations, demographic groups',
        'outcome': 'Named health outcomes, social impacts, measurable results',
    },
    'Literature Review': {
        'person': 'Specific named authors and scholars (first and/or last name)',
        'concept': 'Named theories, frameworks, paradigms, key technical terms',
        'method': 'Named research methods and analytical approaches',
        'discipline': 'Named academic fields and subfields',
    },
}

preset_dropdown = widgets.Dropdown(
    options=list(PRESETS.keys()) + ['Custom'],
    value='General Research',
    description='Preset:',
    style=style, layout=layout
)

custom_area = widgets.Textarea(
    value='',
    placeholder='One entity type per line, optionally with a description after a colon:\nperson: Specific named individuals\nconcept: Named theories and frameworks',
    layout=widgets.Layout(width='100%', height='120px')
)

custom_box = widgets.VBox([widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em;">Enter entity types, one per line. Add a colon and description for better accuracy.</p>'), custom_area])
custom_box.layout.display = 'none'

def on_preset_change(change):
    custom_box.layout.display = '' if change['new'] == 'Custom' else 'none'
preset_dropdown.observe(on_preset_change, names='value')

confidence_slider = widgets.FloatSlider(value=0.5, min=0.1, max=0.99, step=0.05, description='Min confidence:', style=style)
conf_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">Lower values find more entities but may include false positives. 0.5 is a good starting point.</p>')

config_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Entity Types</h3>')

display(widgets.VBox([
    config_header,
    preset_dropdown,
    custom_box,
    confidence_slider,
    conf_help,
]))

## Extract Entities

Run GLiNER2 across your loaded documents. Processing time depends on corpus size and document length.

In [ ]:
# Run Entity Extraction

extract_btn = widgets.Button(description='Extract Entities', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
progress = widgets.HTML(value='')
extract_out = widgets.Output()

entity_df = None
freq_df = None

def on_extract(_):
    global entity_df, freq_df
    extract_out.clear_output()
    progress.value = ''

    if not documents:
        with extract_out:
            print("\u26a0\ufe0f No documents loaded. Run the Data Input cell first.")
        return

    # Get entity types
    if preset_dropdown.value == 'Custom':
        entity_types = {}
        for line in custom_area.value.strip().split('\n'):
            line = line.strip()
            if not line: continue
            if ':' in line:
                name, desc = line.split(':', 1)
                entity_types[name.strip()] = desc.strip()
            else:
                entity_types[line] = line
        if not entity_types:
            with extract_out:
                print("\u26a0\ufe0f Please define at least one entity type.")
            return
    else:
        entity_types = PRESETS[preset_dropdown.value]

    min_conf = confidence_slider.value

    with extract_out:
        print(f"Extracting entities from {len(documents)} segments...")
        print(f"   Entity types: {list(entity_types.keys())}")
        print(f"   Min confidence: {min_conf}")
        print()

    all_rows = []
    total = len(documents)

    for idx, (doc, name) in enumerate(zip(documents, doc_names)):
        if (idx + 1) % 10 == 0 or idx == total - 1:
            progress.value = f'<span style="color: #274C77;">{idx+1}/{total} segments processed</span>'

        chunks = chunk_text(doc)
        for chunk in chunks:
            try:
                result = extractor.extract_entities(chunk, entity_types, include_confidence=True)
                entities = result.get('entities', {})
                for etype, ents in entities.items():
                    for e in ents:
                        if isinstance(e, dict):
                            conf = e.get('confidence', 0)
                            text_val = normalize_text(e.get('text', '')).strip()
                            if conf >= min_conf and len(text_val) > 1 and text_val != '+':
                                # Get context snippet (surrounding words)
                                ent_pos = chunk.find(text_val)
                                if ent_pos >= 0:
                                    start = max(0, ent_pos - 60)
                                    end = min(len(chunk), ent_pos + len(text_val) + 60)
                                    context = chunk[start:end].strip()
                                    if start > 0: context = '...' + context
                                    if end < len(chunk): context = context + '...'
                                else:
                                    context = ''

                                # Source file name (strip _N suffix)
                                source_file = name.rsplit('_', 1)[0] if '_' in name else name

                                all_rows.append({
                                    'entity': text_val,
                                    'type': etype,
                                    'confidence': round(conf, 3),
                                    'source': source_file,
                                    'context': context,
                                })
            except Exception:
                pass

    progress.value = ''

    if not all_rows:
        with extract_out:
            print("\u26a0\ufe0f No entities found. Try lowering the confidence threshold or using different entity types.")
        return

    entity_df = pd.DataFrame(all_rows)

    # Case-normalized frequency table
    entity_df['entity_normalized'] = entity_df['entity'].str.strip()
    # Group by lowercase to merge case variants, keep the most common casing
    case_map = {}
    for ent in entity_df['entity_normalized']:
        key = ent.lower()
        case_map.setdefault(key, []).append(ent)
    canonical = {}
    for key, variants in case_map.items():
        from collections import Counter as _Counter
        canonical[key] = _Counter(variants).most_common(1)[0][0]
    entity_df['entity_canonical'] = entity_df['entity_normalized'].str.lower().map(canonical)

    freq_df = entity_df.groupby(['entity_canonical', 'type']).agg(
        mentions=('source', 'size'),
        avg_confidence=('confidence', 'mean'),
        sources=('source', lambda x: ', '.join(sorted(set(x)))),
        n_sources=('source', 'nunique'),
    ).reset_index().rename(columns={'entity_canonical': 'entity'}).sort_values('mentions', ascending=False)

    with extract_out:
        print(f"\u2713 Extracted {len(entity_df)} entity mentions ({len(freq_df)} unique)")
        print()

        # --- Summary by type ---
        print("Entities by type:")
        for etype in sorted(entity_df['type'].unique()):
            subset = freq_df[freq_df['type'] == etype]
            total_mentions = subset['mentions'].sum()
            print(f"   {etype}: {total_mentions} mentions ({len(subset)} unique)")
        print()

        # --- Bar chart: top entities per type ---
        types = sorted(entity_df['type'].unique())
        n_types = len(types)
        fig, axes = plt.subplots(1, n_types, figsize=(5 * n_types, max(4, min(8, len(freq_df) // n_types * 0.3))))
        if n_types == 1: axes = [axes]

        for ax, etype in zip(axes, types):
            subset = freq_df[freq_df['type'] == etype].head(12)
            if subset.empty: continue
            entities = subset['entity'].tolist()[::-1]
            counts = subset['mentions'].tolist()[::-1]
            ax.barh(entities, counts, color='#6096BA', edgecolor='#274C77', linewidth=0.5)
            ax.set_title(etype.title(), fontsize=12, color='#274C77', fontweight='bold')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#E7ECEF')
            ax.spines['bottom'].set_color('#E7ECEF')
            ax.tick_params(colors='#8B8C89', labelsize=8)
            for bar, count in zip(ax.patches, counts):
                ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, str(count), va='center', fontsize=8, color='#274C77')

        plt.suptitle('Top Entities by Type', fontsize=14, color='#274C77', fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

        # --- Source-by-Entity heatmap (top 20 entities) ---
        top_entities = freq_df.head(20)['entity'].tolist()
        source_entity_data = {}
        for _, row in entity_df[entity_df['entity_canonical'].isin(top_entities)].iterrows():
            src = row['source']
            ent = row['entity_canonical']
            source_entity_data.setdefault(src, {})
            source_entity_data[src][ent] = source_entity_data[src].get(ent, 0) + 1

        if source_entity_data and len(source_entity_data) > 1:
            print()
            hm_df = pd.DataFrame(source_entity_data).T.fillna(0).astype(int)
            # Sort columns by total mentions, rows by total
            col_order = hm_df.sum().sort_values(ascending=False).index
            hm_df = hm_df[col_order]
            hm_df['_total'] = hm_df.sum(axis=1)
            hm_df = hm_df.sort_values('_total', ascending=False).drop('_total', axis=1)

            fig, ax = plt.subplots(figsize=(max(8, len(hm_df.columns) * 0.8), max(3, len(hm_df) * 0.5)))
            im = ax.imshow(hm_df.values, cmap='Blues', aspect='auto')
            ax.set_xticks(range(len(hm_df.columns)))
            ax.set_xticklabels(hm_df.columns, fontsize=8, rotation=45, ha='right')
            ax.set_yticks(range(len(hm_df.index)))
            short_names = [n[:35] + '...' if len(n) > 35 else n for n in hm_df.index]
            ax.set_yticklabels(short_names, fontsize=8)
            for i in range(len(hm_df.index)):
                for j in range(len(hm_df.columns)):
                    val = hm_df.values[i, j]
                    if val > 0:
                        color = 'white' if val > hm_df.values.max() * 0.5 else '#274C77'
                        ax.text(j, i, str(int(val)), ha='center', va='center', fontsize=7, color=color, fontweight='bold')
            ax.set_title('Top Entities by Source File', fontsize=13, color='#274C77', fontweight='bold', pad=15)
            plt.colorbar(im, ax=ax, shrink=0.8)
            plt.tight_layout()
            plt.show()

        # --- Top entities table with context ---
        print()
        print("Top entities with context:")
        display(freq_df[['entity', 'type', 'mentions', 'n_sources', 'avg_confidence', 'sources']].head(25).rename(
            columns={'avg_confidence': 'avg_conf', 'n_sources': 'files'}))

extract_btn.on_click(on_extract)
display(widgets.VBox([extract_btn, progress, extract_out]))

## Export Results

Export entity data as CSV or Excel. The export includes all entity mentions and a frequency summary sheet.

In [ ]:
# Export Results

if entity_df is None:
    print("\u26a0\ufe0f Run entity extraction first.")
else:
    export_format = widgets.Dropdown(options=[('Excel (.xlsx) \u2014 full report', 'xlsx'), ('CSV (.csv)', 'csv')], value='xlsx', description='Format:', style=style, layout=layout)
    export_btn = widgets.Button(description='Export Results', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
    export_out = widgets.Output()

    def on_export(_):
        export_out.clear_output()
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            base = f'entities_{timestamp}'
            fmt = export_format.value

            # Mentions sheet with context
            mentions_df = entity_df[['entity_canonical', 'type', 'confidence', 'source', 'context']].rename(
                columns={'entity_canonical': 'entity'}).sort_values(['type', 'entity'])

            if fmt == 'xlsx':
                filepath = f'{base}.xlsx'
                with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                    freq_df[['entity', 'type', 'mentions', 'n_sources', 'avg_confidence', 'sources']].to_excel(
                        writer, sheet_name='Frequency Summary', index=False)
                    mentions_df.to_excel(writer, sheet_name='All Mentions', index=False)
                style_excel(filepath)
            else:
                filepath = f'{base}_frequency.csv'
                freq_df.to_csv(filepath, index=False)
                mentions_path = f'{base}_mentions.csv'
                mentions_df.to_csv(mentions_path, index=False)

            with export_out:
                print(f"\u2713 Saved: {filepath}")
                print(f"   {len(freq_df)} unique entities, {len(mentions_df)} mentions")
                if IN_COLAB:
                    colab_files.download(filepath)
                    if fmt == 'csv':
                        colab_files.download(mentions_path)

        except Exception as e:
            with export_out:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)
    display(widgets.VBox([export_format, export_btn, export_out]))